# 🎙️ BookVoice-AI — GPU Studio

**KI Hörbuch-Generierung mit NVIDIA T4 GPU (kostenlos)**

---

## 🚀 Schnellstart

1. **Runtime → Change runtime type → T4 GPU** aktivieren
2. **Strg+F9** (Alle Zellen ausführen)
3. Warten bis Link erscheint (~5 Minuten)
4. Link in BookVoice-AI GUI eingeben

---

> ⚠️ Kostenlose Colab-Session läuft ~3-4 Stunden. Danach neu starten.

In [ ]:
# ── Schritt 1: GPU prüfen ──────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU verfügbar!')
    print(result.stdout.split('\n')[8])
else:
    print('❌ Keine GPU! Bitte Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Schritt 2: System-Pakete installieren ──────────
print('📦 Installiere System-Pakete...')
!apt-get install -y ffmpeg calibre tesseract-ocr tesseract-ocr-tur tesseract-ocr-deu -q
print('✅ System-Pakete installiert!')

In [ ]:
# ── Schritt 3: Python-Pakete installieren ──────────
print('🐍 Installiere Python-Pakete...')
!pip install -q coqui-tts
!pip install -q fastapi uvicorn python-multipart pytesseract pdf2image PyPDF2 pillow ebooklib edge-tts
print('✅ Python-Pakete installiert!')

In [ ]:
# ── Schritt 4: BookVoice-AI herunterladen ──────────
import urllib.request
print('⬇️ Lade BookVoice-AI...')
url = 'https://raw.githubusercontent.com/dolunay38/BookVoice-AI/main/tts_server.py'
urllib.request.urlretrieve(url, 'tts_server.py')
print('✅ tts_server.py heruntergeladen!')

In [ ]:
# ── Schritt 5: Ordner anlegen ──────────────────────
import os
dirs = ['/content/HOERBUCH', '/content/tts_models', '/content/musik', '/content/covers']
for d in dirs:
    os.makedirs(d, exist_ok=True)
print('✅ Ordner erstellt!')

In [ ]:
# ── Schritt 6: XTTS-v2 Modell herunterladen ────────
print('🔄 Lade XTTS-v2 Modell (~1.8 GB)...')
print('   Bitte warten — dauert 3-5 Minuten...')
from TTS.api import TTS
import os
os.environ['COQUI_TOS_AGREED'] = '1'
tts_check = TTS('tts_models/multilingual/multi-dataset/xtts_v2')
print('✅ XTTS-v2 Modell bereit!')

In [ ]:
# ── Schritt 7: Standard-Stimme erstellen ───────────
import numpy as np
import torch
import torchaudio

voice_path = '/content/tts_models/stimme.wav'
if not os.path.exists(voice_path):
    sample_rate = 22050
    t = np.linspace(0, 3, int(sample_rate * 3))
    audio = np.sin(2 * np.pi * 200 * t) * 0.3
    audio_tensor = torch.tensor(audio, dtype=torch.float32).unsqueeze(0)
    torchaudio.save(voice_path, audio_tensor, sample_rate)
    print('✅ Standard-Stimme erstellt!')
else:
    print('✅ Stimme bereits vorhanden!')

In [ ]:
# ── Schritt 8+9: Server starten + Tunnel ──────────
import subprocess, os, time, re

# Server starten
env = os.environ.copy()
env['TTS_OUTPUT'] = '/content/HOERBUCH'
env['TTS_LANG'] = 'tr'
env['COQUI_TOS_AGREED'] = '1'

server = subprocess.Popen(
    ['uvicorn', 'tts_server:app', '--host', '0.0.0.0', '--port', '7500'],
    env=env
)
print('⏳ Server startet...')
time.sleep(5)
print('✅ Server läuft!')

# Cloudflared installieren
print('📦 Installiere Cloudflared...')
subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-O', '/usr/local/bin/cloudflared'], capture_output=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'])
print('✅ Cloudflared bereit!')

# Tunnel starten
proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:7500'],
    stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True
)

print('⏳ Warte auf URL...')
for i in range(30):
    line = proc.stderr.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            print('='*60)
            print('🎉 BookVoice-AI GPU Studio ist bereit!')
            print('='*60)
            print(f'\n🔗 URL: {url}')
            print('\n📋 So nutzen:')
            print('   1. URL kopieren')
            print('   2. BookVoice-AI GUI → ☁️ GPU Colab')
            print('   3. URL eingeben → Verbinden')
            print('   4. Hörbuch mit GPU generieren!')
            print('\n⚠️  Session läuft ~3-4 Stunden')
            print('='*60)
            break
    time.sleep(1)


In [ ]:
# ── Optional: Eigene Stimme hochladen ──────────────
from google.colab import files
print('📁 Stimme hochladen (WAV/MP3, 30-60 Sek):')
uploaded = files.upload()
for filename, data in uploaded.items():
    save_path = f'/content/tts_models/{filename}'
    with open(save_path, 'wb') as f:
        f.write(data)
    print(f'✅ Stimme gespeichert: {save_path}')

In [ ]:
# ── Optional: Fertige Hörbücher herunterladen ──────
from google.colab import files
import glob

hoerbucher = glob.glob('/content/HOERBUCH/**/*.mp3', recursive=True) + \
             glob.glob('/content/HOERBUCH/**/*.m4b', recursive=True)

if hoerbucher:
    print(f'📚 {len(hoerbucher)} Hörbuch(er) gefunden:')
    for h in hoerbucher:
        print(f'   {h}')
        files.download(h)
else:
    print('Noch keine Hörbücher generiert.')